# GeoLogic agents with Qwen3.5-9B: three setups

This notebook runs Qwen3.5-9B (served locally with vLLM) on GeoLogic map questions under three setups and compares them on the same items.

| Setup | Environment | Tools offered | What a tool call returns |
|---|---|---|---|
| **1. Compositional agent** | `geo_env.Env` | 7 survey verbs: `find`, `neighbors`, `draw`, `intersect`, `merge`, `measure`, `sort` | **An answer** (e.g. *"Regions that share an edge with Region B: A, C, D."*) plus the re-rendered diagram |
| **2. Annotation agent** | `geo_env_annotation.AnnotationEnv` | Markup tools: `label_vertex`, `label_angle`, `label_edge`, `highlight_region`, `union`, `draw_line`, `undo` | **Only a confirmation** (e.g. *"Highlighted Region B."*) plus the re-rendered diagram. Every fact must be read off the image. |
| **3. No tool use** | `geo_env.Env` | None. No tool schema is sent and the system prompt never mentions tools. | n/a: image-only baseline |

In every setup the model sees the same 460×460 diagram a human participant saw. Setups 1 and 2 are tool-use episodes; setup 3 is the control. The no-tool prompt is identical in `geo_agent.py` and `geo_agent_annotation.py`, so it is the shared baseline for both tool arms.

**Notebook outline**

0. Environment, paths, and the vLLM server
1. Sampling parameters (Qwen3.5 thinking mode, VL)
2. Dataset and a shared episode runner
3. Setup 1: compositional agent
4. Setup 2: annotation agent
5. Setup 3: no tool use
6. Full dataset run over all three setups
7. Shut down, plus known caveats

## Vertex pilot compatibility
This copy reads each item’s `structure_complexity` and verifies the reconstructed labeled geometry. It loads the frozen generator/tools/agent sources in this package. No model evaluation was run during dataset creation.
The normal turn budget is set to 10. The previously identified extra tools-disabled final-request fix remains deferred in the frozen agents. The inherited tutorial scorer still uses strict normalized-string equality; unordered region lists/pairs may need semantic scoring for evaluation.


## 0. Environment and paths

Edit the paths in this cell for your machine. Everything else in the notebook reads from these variables.

Python files expected on `sys.path`: `geo_env.py`, `geo_agent.py`, `geo_env_annotation.py`, `geo_agent_annotation.py`, `visual_tools.py`, `visual_tools_annotation.py`, plus the generator modules (`Graph`, `BuildRandomMap`, `DrawGraph`, `map_helpers`, `tools_human`) inside the repo.

In [ ]:
import os, sys, json, datetime, traceback
from pathlib import Path

# ---- cluster paths (edit these) ----
HF_HOME  = os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface"))
VENV_DIR = os.environ.get("VIRTUAL_ENV", sys.prefix)
VENV_PY  = f"{VENV_DIR}/bin/python"
MODEL_ID = "Qwen/Qwen3.5-9B"
PORT     = int(os.environ.get("VLLM_PORT", "8000"))

# ---- repo + dataset (unchanged) ----
# Open from the package folder, the data folder, or geo_agent/.
_candidates = [Path.cwd(), Path.cwd() / "vertex_pilot_v1", Path.cwd().parent / "vertex_pilot_v1"]
REPO = next((p.resolve() for p in _candidates if (p / "runner_compat.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Open this notebook from vertex_pilot_v1 or its parent folder")
DATASET_PATH = REPO / "dataset" / "dataset.json"

os.environ["HF_HOME"] = HF_HOME
for p in [REPO / "generator", REPO / "tools", REPO / "agent", REPO]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# The agents talk to vLLM through the OpenAI client.
os.environ["OPENAI_BASE_URL"] = f"http://localhost:{PORT}/v1"
os.environ["OPENAI_API_KEY"] = "EMPTY"

print("repo   :", REPO)
print("dataset:", DATASET_PATH, "(exists)" if DATASET_PATH.exists() else "(MISSING)")
# Prevent accidentally using an already imported, older generator.
import runner_compat
runner_compat.verify_loaded_sources()


### 0a. Launch the vLLM server

Starts `vllm.entrypoints.openai.api_server` in the background and logs to `/tmp/vllm.log`. The `qwen3` reasoning parser separates thinking from the final answer, and the `qwen3_coder` tool-call parser enables tool use. Run this once per session; skip it if a server is already up on `PORT`.

In [ ]:
import subprocess
from huggingface_hub import snapshot_download

MODEL_PATH = os.environ.get("MODEL_PATH")
if not MODEL_PATH or not Path(MODEL_PATH).is_dir():
    raise ValueError("Set MODEL_PATH to your local Qwen3.5-9B model directory")
print("model path:", MODEL_PATH)

LOG = "/tmp/vllm.log"

env_vars = os.environ.copy()
env_vars.pop("LD_PRELOAD", None)
env_vars.pop("OPENAI_API_KEY", None)
env_vars["LD_LIBRARY_PATH"] = ":".join(
    p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":")
    if p and "gcc/15.1.0" not in p
)
env_vars.update({
    "HF_HOME": HF_HOME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "VLLM_NO_USAGE_STATS": "1",
    "DO_NOT_TRACK": "1",
    "VIRTUAL_ENV": VENV_DIR,
    "PATH": f"{VENV_DIR}/bin:" + env_vars["PATH"],
    "VLLM_USE_FLASHINFER_SAMPLER": "0",
})

server = subprocess.Popen(
    [VENV_PY, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL_PATH,
     "--served-model-name", MODEL_ID,
     "--port", str(PORT),
     "--tensor-parallel-size", "1",
     "--max-model-len", "65536",
     "--gpu-memory-utilization", "0.90",
     "--limit-mm-per-prompt", '{"image": 2}',
     "--reasoning-parser", "qwen3",
     "--enable-auto-tool-choice",
     "--tool-call-parser", "qwen3_coder"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env_vars,
)
print("vLLM pid", server.pid, "| log:", LOG)

In [ ]:
# Wait for the server to report healthy (up to 15 minutes).
import time, requests

for i in range(180):
    if server.poll() is not None:
        print("SERVER DIED. Last 80 lines of the log:")
        print("".join(open(LOG).readlines()[-80:]))
        break
    try:
        if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
            print("server up after", i * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
else:
    print("timed out; check", LOG)

## 1. Sampling parameters: Qwen3.5 thinking mode for VL tasks

All three setups use the Qwen3.5 recommended settings for **thinking mode on VL / precise coding tasks**:

| param | value |
|---|---|
| `temperature` | 0.6 |
| `top_p` | 0.95 |
| `top_k` | 20 |
| `min_p` | 0.0 |
| `presence_penalty` | 0.0 |
| `repetition_penalty` | 1.0 |

**How they are applied.** `geo_agent.py` defines module-level `SAMPLING` (sent as standard OpenAI arguments) and `EXTRA_BODY` (vLLM-only arguments). Its defaults are the general thinking-mode values (`temperature=1.0`, `presence_penalty=1.5`), so this notebook overrides them. `temperature`, `top_p`, and `presence_penalty` are standard OpenAI fields and go in `SAMPLING`; `top_k`, `min_p`, and `repetition_penalty` are vLLM extensions and go in `extra_body`. `max_tokens` keeps the value from `geo_agent.py`.

`geo_agent_annotation.py` imports `SAMPLING` and `EXTRA_BODY` *by name* from `geo_agent`, so it holds its own reference. `apply_qwen_sampling()` sets both modules explicitly, and the episode runner below calls it before every episode, so a stray `importlib.reload` cannot silently restore the old defaults.

Thinking is on for every run (`THINKING = True`), which is what these parameters are tuned for.

In [ ]:
import importlib
import pandas as pd
from IPython.display import display

import geo_env, geo_agent
import geo_env_annotation as gea
import geo_agent_annotation as gaa

THINKING = True

QWEN_SAMPLING = dict(          # standard OpenAI chat.completions arguments
    temperature=0.6,
    top_p=0.95,
    presence_penalty=0.0,
)
QWEN_EXTRA_BODY = dict(        # vLLM-specific sampling arguments
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
)
MAX_TOKENS = 32768   # new experiment; preserve the original 10024-token results


def apply_qwen_sampling(verbose=False):
    # Rebind the globals that each module's run_episode reads at call time.
    for mod in (geo_agent, gaa):
        mod.SAMPLING = {**QWEN_SAMPLING, "max_tokens": MAX_TOKENS}
        mod.EXTRA_BODY = dict(QWEN_EXTRA_BODY)
    if verbose:
        for mod in (geo_agent, gaa):
            print(f"{mod.__name__:22s} SAMPLING={mod.SAMPLING}  EXTRA_BODY={mod.EXTRA_BODY}")


apply_qwen_sampling(verbose=True)

## 2. Dataset and a shared episode runner

The dataset is `vertex_pilot_v1/dataset/dataset.json`. Each entry in `items` carries `item_id`, `region_count`, `diagram_seed`, and a `question` dict with `question_text` and the gold `answer`. The diagram is regenerated from `(region_count, diagram_seed, structure_complexity)` using frozen sources; a geometry hash is checked before each episode.

`run_item(setup, item)` is the single entry point for all three setups. It builds a **fresh** environment for every episode, because `merge`, `union`, and `draw` mutate the map and would otherwise leak between runs. It then runs the episode and saves a JSON record containing the final answer, the gold answer, every tool call (arguments, output, and any swallowed error), the full message history with images replaced by placeholders, and the final diagram as a PNG.

The tool-use setups take a `tool_mode` flag:

- `"auto"`: tools are offered and the model decides whether to use them
- `"required"`: the first `force_first_n` turns must contain a tool call
- `"none"`: no tools; this is what setup 3 uses

In [ ]:
graph = json.load(open(DATASET_PATH))
DATASET_ID = graph.get("dataset_id") or graph.get("id") or "geologic_all_hard_24"
ITEMS = graph["items"]
print(f"{DATASET_ID}: {len(ITEMS)} items")
pd.DataFrame([{"item_id": it["item_id"],
               "regions": it["region_count"],
               "seed": it["diagram_seed"],
               "question": it["question"]["question_text"].strip(),
               "gold": it["question"].get("answer")} for it in ITEMS[:5]])

In [ ]:
import time

# ---------------- shared configuration ----------------
DIAGRAM_COMPLEXITY = "high"   # "current" or "high"; used by all three setups
ADDRESSING = "descriptor"        # annotation agent only: "descriptor" or "coordinate"
MAX_TURNS = 10
RUN_ROOT = Path("runs") / datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

SETUPS = ("compositional", "annotation", "no_tools")


def make_env(setup, item):
    return runner_compat.make_verified_env(setup, item, addressing=ADDRESSING)


def _strip_images(messages):
    # Drop base64 image payloads so the saved history stays readable.
    out = []
    for m in messages:
        m = dict(m)
        if isinstance(m.get("content"), list):
            m["content"] = [
                {"type": "image", "note": "[diagram omitted]"}
                if isinstance(c, dict) and c.get("type") in ("image_url", "image") else c
                for c in m["content"]
            ]
        out.append(m)
    return out


def run_item(setup, item, tool_mode="auto", force_first_n=1,
             max_turns=MAX_TURNS, verbose=1):
    # Run one dataset item under one setup; return (record, result).
    apply_qwen_sampling()
    q = item["question"]
    item_structure = item.get("structure_complexity", DIAGRAM_COMPLEXITY)
    effective_mode = "none" if setup == "no_tools" else tool_mode
    tag = setup if setup == "no_tools" else f"{setup}_{tool_mode}"
    run_id = f"{DATASET_ID}_c{item_structure}_{tag}_{item['item_id']}"
    outdir = RUN_ROOT / tag
    (outdir / "images").mkdir(parents=True, exist_ok=True)

    record = {
        "run_id": run_id, "dataset_id": DATASET_ID, "item_id": item["item_id"],
        "setup": setup, "tool_mode": effective_mode,
        "addressing": ADDRESSING if setup == "annotation" else None,
        "complexity": item_structure,
        "question_complexity": item.get("question_complexity"),
        "region_count": item["region_count"], "diagram_seed": item["diagram_seed"],
        "question_id": q.get("question_id") or q.get("id"),
        "question_type": q.get("question_type") or q.get("type"),
        "question_text": q["question_text"],
        "record_schema": "geo_raw_episode_v2",
        "max_turns": max_turns,
        "model_id": MODEL_ID,
        "geometry_sha256": item.get("geometry_sha256"),
        "dataset_sha256": __import__("hashlib").sha256(DATASET_PATH.read_bytes()).hexdigest(),
        "sampling": {**QWEN_SAMPLING, **QWEN_EXTRA_BODY, "max_tokens": MAX_TOKENS,
                     "thinking": THINKING},
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    }

    t0 = time.time()
    result = None
    try:
        env = make_env(setup, item)
        agent = gaa if setup == "annotation" else geo_agent
        capture_kwargs = {}
        if setup != "annotation":
            response_log_path = outdir / "responses" / f"{run_id}.jsonl"
            capture_kwargs["response_log_path"] = response_log_path
            record["response_log_path"] = str(response_log_path)
        initial_path = outdir / "images" / f"{run_id}_initial.png"
        env.render().save(initial_path)
        record["initial_image_path"] = str(initial_path)
        record["system_prompt"] = agent._SYSTEM_PROMPTS[effective_mode]
        record["tool_schemas"] = geo_agent.TOOLS if effective_mode != "none" and setup == "compositional" else []
        result = agent.run_episode(
            env, q["question_text"], max_turns=max_turns, thinking=THINKING,
            verbose=verbose, tool_mode=effective_mode, force_first_n=force_first_n, **capture_kwargs)

        img_path = ""
        if result.get("final_image") is not None:
            img_path = str(outdir / "images" / f"{run_id}.png")
            result["final_image"].save(img_path)

        record.update({
            "answer": result["answer"],
            "status": result.get("status"),
            "incomplete_reason": result.get("incomplete_reason"),
            "finish_reason": result.get("finish_reason"),
            "requests": result.get("requests", []),
            "usage_totals": result.get("usage_totals", {}),
            "n_turns": result["turns"],
            "n_tool_calls": len(result["trace"]),
            "tool_sequence": [t["tool"] for t in result["trace"]],
            "trace": result["trace"],                        # tool, args, output
            "env_tool_calls": list(getattr(env, "tool_calls", [])),  # includes errors
            "program": list(getattr(env, "program", [])),
            "n_tool_errors": sum(1 for tc in getattr(env, "tool_calls", []) if tc.get("error")),
            "raw_final": result["raw"],
            "messages": _strip_images(result["messages"]),
            "note": result.get("note"),
            "image_path": img_path,
            "run_error": None,
        })
    except Exception as e:
        print("!! FAILED:", repr(e))
        record.update({"answer": None, "status": "runtime_error",
                       "run_error": f"{e!r}\n{traceback.format_exc()}"})

    record["elapsed_s"] = round(time.time() - t0, 2)
    record_path = outdir / f"{run_id}.json"
    temporary_path = record_path.with_suffix(".json.tmp")
    with open(temporary_path, "w") as f:
        json.dump(record, f, indent=2, default=str)
    temporary_path.replace(record_path)
    return record, result


def summarize(record):
    print(f"\nRESPONSE: {record.get('answer')!r}")
    print("status:", record.get("status"), "| reason:", record.get("incomplete_reason"), "| finish:", record.get("finish_reason"))
    print(f"tools: {record.get('tool_sequence')} | turns: {record.get('n_turns')} "
          f"| tool errors: {record.get('n_tool_errors')}")
    if record.get("program"):
        print("program:")
        for i, line in enumerate(record["program"], 1):
            print(f"  {i}. {line}")

For the single-item demos in sections 3 to 5 we use one dataset item so the three setups answer exactly the same question about exactly the same diagram. Change `DEMO_INDEX` to try another item.

In [ ]:
DEMO_INDEX = 0
demo_item = ITEMS[DEMO_INDEX]
print(demo_item["item_id"], "|", demo_item["question"]["question_text"].strip())
print("gold:", demo_item["question"].get("answer"))
display(make_env("compositional", demo_item).render())

## 3. Setup 1: Compositional agent

The model gets the seven survey verbs from `geo_agent.TOOLS`. These tools **compute**: `measure`, `neighbors`, `sort`, and the others return facts about the map in text, and the diagram is re-rendered with the selection shaded. Vertices and lines found or drawn by earlier calls are given names (`v1`, `L1`, ...) that later calls can reference, which is what makes the tools compositional. `merge` creates a new first-class region `U` that can be measured like any other.

### 3a. Try the tools by hand (no model)

`geo_agent.dispatch(env, tool, args)` is exactly what the episode loop calls when the model emits a tool call.

In [ ]:
env = make_env("compositional", demo_item)
for tool, args in [
    ("measure",   {"what": "regions"}),
    ("neighbors", {"obj": "B", "kind": "edge"}),
    ("measure",   {"what": "edge_count", "regions": ["A"]}),
    ("find",      {"objs": ["A"], "which": "leftmost"}),
]:
    print(f">>> {tool}({args})")
    print("   ", geo_agent.dispatch(env, tool, args))
display(env.render())

### 3b. Run the compositional agent on the demo item

`verbose=1` prints tool calls and outputs, `2` adds the model's visible text each turn, and `3` adds the thinking trace and raw message.

In [ ]:
comp_record, comp_result = run_item("compositional", demo_item, tool_mode="auto", verbose=2)
summarize(comp_record)

## 4. Setup 2: Annotation agent

The model gets markup tools from `geo_agent_annotation.build_tools(ADDRESSING)`. These tools **only draw**. Each one returns a confirmation sentence and nothing about the map, so the model must look at the re-rendered diagram to learn anything. This makes the annotation episode a visual-reasoning task with a markup aid rather than a tool-use task, and the system prompt tells the model so.

Unlike the compositional `merge`, the annotation `union` is visual only: it hides the shared border, but the original regions stay addressable.

**Vertex addressing** (`ADDRESSING` in section 2):

- `"descriptor"` (default): name a vertex by a region and a corner property, e.g. `{"region": "B", "which": "leftmost"}`, or by the regions that meet there, e.g. `{"regions": ["A", "B", "C"]}`.
- `"coordinate"`: give normalized `x`, `y` in [0, 1] and the nearest vertex is used. This is closer to a mouse click, but VLMs localize poorly, so expect misses.

### 4a. Try the tools by hand (no model)

Note that the outputs say nothing about the map.

In [ ]:
env = make_env("annotation", demo_item)
print("tools offered:", [t["function"]["name"] for t in gaa.build_tools(ADDRESSING)])
vertex = ({"region": "B", "which": "leftmost"} if ADDRESSING == "descriptor"
          else {"x": 0.15, "y": 0.85})
for tool, args in [
    ("highlight_region", {"region": "B"}),
    ("label_vertex", vertex),
    ("label_edge", {"edge": ["A", "B"]}),
]:
    print(f">>> {tool}({args})")
    print("   ", gaa.dispatch(env, tool, args))
    err = env.tool_calls[-1].get("error") if env.tool_calls else None
    if err:
        print("    !! error:", err)   # e.g. A and B may not share an edge on this map
display(env.render())

### 4b. Run the annotation agent on the demo item

In [ ]:
ann_record, ann_result = run_item("annotation", demo_item, tool_mode="auto", verbose=2)
summarize(ann_record)

## 5. Setup 3: No tool use

The image-only baseline. No tool schema is sent and the system prompt never mentions tools, so the model must answer from the diagram alone in a single turn. If the model emits a stray tool call anyway, it is ignored and its text is treated as the answer.

In [ ]:
none_record, none_result = run_item("no_tools", demo_item, verbose=2)
summarize(none_record)

In [ ]:
# Side-by-side comparison of the three demo runs.
pd.DataFrame([comp_record, ann_record, none_record])[
    ["setup", "tool_mode", "answer", "status", "incomplete_reason",
     "n_turns", "n_tool_calls", "n_tool_errors", "elapsed_s"]]

### Optional: forcing tool use

For either tool setup, `tool_mode="required"` forces a tool call on the first `force_first_n` turns and then lets the model answer. It is useful for checking whether a tool helps when the model would otherwise skip it.

In [ ]:
# forced_record, _ = run_item("compositional", demo_item, tool_mode="required",
#                             force_first_n=1, max_turns=8, verbose=1)
# summarize(forced_record)

## 6. Full dataset run

Runs every selected item under every selected setup. Each episode writes its own JSON record to `runs/<timestamp>/<setup>/`, and the combined results are saved as `all_runs.jsonl`, `summary.csv`, and `summary.parquet` in `RUN_ROOT`.

Set `N = None` to run the whole dataset. With thinking on, each episode can take a while, so start small.

In [ ]:
# ---------------- config ----------------
N = None                # run all 74 items
START = 0               # skip ahead
RUN_SETUPS = ("compositional", "no_tools")  # agreed pilot conditions
TOOL_MODE = "auto"      # for the two tool setups: "auto" or "required"
VERBOSE = 0             # 0 = quiet; 1+ prints each episode

items = ITEMS[START:] if N is None else ITEMS[START:START + N]
all_records = []

for setup in RUN_SETUPS:
    print(f"\n{'=' * 70}\n{setup.upper()}  ({len(items)} items)\n{'=' * 70}")
    for i, item in enumerate(items, 1):
        rec, _ = run_item(setup, item, tool_mode=TOOL_MODE, verbose=VERBOSE)
        all_records.append(rec)
        print(f"[{i}/{len(items)}] {item['item_id']}: {rec.get('answer')!r} "
              f"(status={rec.get('status')}, "
              f"{rec.get('n_tool_calls', 0)} tool calls)")

# ---------------- save ----------------
RUN_ROOT.mkdir(parents=True, exist_ok=True)
with open(RUN_ROOT / "all_runs.jsonl", "w") as f:
    for rec in all_records:
        f.write(json.dumps(rec, default=str) + "\n")

SUMMARY_COLS = ["run_id", "item_id", "setup", "tool_mode", "addressing", "complexity",
                "question_text", "answer", "n_turns",
                "n_tool_calls", "n_tool_errors", "tool_sequence", "elapsed_s",
                "image_path", "run_error", "status", "incomplete_reason", "finish_reason", "usage_totals"]
summary = pd.DataFrame(all_records).reindex(columns=SUMMARY_COLS)
for c in ("tool_sequence", "usage_totals"):
    summary[c] = summary[c].map(lambda v: json.dumps(v, default=str))
summary.to_csv(RUN_ROOT / "summary.csv", index=False)
summary.to_parquet(RUN_ROOT / "summary.parquet", index=False)
print(f"\nsaved {len(all_records)} episodes -> {RUN_ROOT}")

In [ ]:
# Accuracy is computed offline, never during model execution.
print("Evaluate separately with evaluate_results.py --run-dir <directory>")


`correct` is a normalized exact string match (case and whitespace insensitive). Answers that are right but formatted differently, such as a set of regions listed in another order, will be marked wrong, so spot-check the `summary` table before reporting numbers.

## 7. Shut down and known caveats

In [ ]:
server.terminate()
server.wait(timeout=60)
print("vLLM server stopped")

**Known caveats**

- **Annotation tool wording is not yet verified against the survey UI.** The tool names and descriptions in `geo_agent_annotation.py` were written from `AnnotationSession`'s method signatures, not from what human participants read on screen. Until they are replaced with the UI's own labels, a gap between the annotation and no-tool setups partly reflects prompt wording, not only the affordance.
- **Thinking traces are not saved.** Following the Qwen best practice, `run_episode` keeps `reasoning_content` out of the multi-turn history, so it is not in `messages`. Use `verbose=3` to print it during a run.
- **Only the current diagram is kept in context.** Older renders are replaced with a text placeholder each turn, which matches the server's `--limit-mm-per-prompt` setting.
- **Do not share a map between `geo_env.Env` and `AnnotationEnv`.** They write different label caches onto the same face objects. `make_env` always builds a fresh one.